# AVISTA Validation Study — UCI Obesity Levels Dataset

## Objective

This notebook evaluates AVISTA using the public **UCI Estimation of Obesity Levels Based on Eating Habits and Physical Condition** dataset. The objective is to compare AVISTA with an equivalent direct Jupyter implementation under controlled experimental settings and with AutoGluon as an external AutoML platform.

## Dataset

The dataset contains **2,111 observations**, **16 predictors**, and a seven-class target representing obesity level:

* Insufficient Weight
* Normal Weight
* Overweight Level I
* Overweight Level II
* Obesity Type I
* Obesity Type II
* Obesity Type III

The predictors include both **numerical and categorical variables**, making this dataset useful for validating AVISTA's mixed-type preprocessing pipeline.

UCI reports no missing values. The repository also notes that approximately **77% of the observations were synthetically generated using Weka and SMOTE**, while 23% were collected directly from users. This characteristic is considered when interpreting the validation results.

## Experimental Setup

A fixed random seed of **42** is used throughout. The dataset is divided into:

* **70% training**
* **10% validation**
* **20% test**

For AVISTA and direct Jupyter experiments, preprocessing is fitted on training data only. Numerical missing values, if present, are median-imputed and numerical scaling follows the selected AVISTA configuration. Categorical missing values are represented using `Unknown` and categorical predictors are one-hot encoded.

Five-fold stratified cross-validation is performed within the training partition using leakage-safe preprocessing.

The evaluated models are:

* Logistic Regression
* Random Forest
* XGBoost

Performance is evaluated using **Accuracy, Macro-F1, Macro-Precision, Macro-Recall, and multiclass ROC-AUC**, together with training/effective sample size and runtime.

AutoGluon uses the same raw training, validation, and test observations while retaining its native preprocessing and training behavior.


In [1]:
# ============================================================
# Cell 1: Download and load UCI Obesity dataset
# ============================================================

from ucimlrepo import fetch_ucirepo
import pandas as pd

# Download dataset
obesity = fetch_ucirepo(id=544)

# Features and target
X = obesity.data.features.copy()
y = obesity.data.targets.copy()

# Combined dataframe
df = pd.concat([X, y], axis=1)

print("Dataset loaded successfully.")
print(f"Total rows     : {df.shape[0]}")
print(f"Features       : {X.shape[1]}")
print(f"Target column  : {y.columns[0]}")
print(f"Dataset shape  : {df.shape}")

display(df.head())

print("\n=== VARIABLE INFORMATION ===")
display(obesity.variables)

Dataset loaded successfully.
Total rows     : 2111
Features       : 16
Target column  : NObeyesdad
Dataset shape  : (2111, 17)


,Gender,Age,Height,Weight,family_history_with_overweight,FAVC,FCVC,NCP,CAEC,SMOKE,CH2O,SCC,FAF,TUE,CALC,MTRANS,NObeyesdad
0,Female,21.0,1.62,64.0,yes,no,2.0,3.0,Sometimes,no,2.0,no,0.0,1.0,no,Public_Transportation,Normal_Weight
1,Female,21.0,1.52,56.0,yes,no,3.0,3.0,Sometimes,yes,3.0,yes,3.0,0.0,Sometimes,Public_Transportation,Normal_Weight
2,Male,23.0,1.80,77.0,yes,no,2.0,3.0,Sometimes,no,2.0,no,2.0,1.0,Frequently,Public_Transportation,Normal_Weight
3,Male,27.0,1.80,87.0,no,no,3.0,3.0,Sometimes,no,2.0,no,2.0,0.0,Frequently,Walking,Overweight_Level_I
4,Male,22.0,1.78,89.8,no,no,2.0,1.0,Sometimes,no,2.0,no,0.0,0.0,Sometimes,Public_Transportation,Overweight_Level_II



=== VARIABLE INFORMATION ===


,name,role,type,demographic,description,units,missing_values
0,Gender,Feature,Categorical,Gender,None,None,no
1,Age,Feature,Continuous,Age,None,None,no
2,Height,Feature,Continuous,None,None,None,no
3,Weight,Feature,Continuous,None,None,None,no
4,family_history_with_overweight,Feature,Binary,None,Has a family member suffered or suffers from o...,None,no
5,FAVC,Feature,Binary,None,Do you eat high caloric food frequently?,None,no
6,FCVC,Feature,Integer,None,Do you usually eat vegetables in your meals?,None,no
7,NCP,Feature,Continuous,None,How many main meals do you have daily?,None,no
8,CAEC,Feature,Categorical,None,Do you eat any food between meals?,None,no
9,SMOKE,Feature,Binary,None,Do you smoke?,None,no


In [2]:
# ============================================================
# Cell 2: Define feature types
# ============================================================

TARGET = "NObeyesdad"

numerical_features = [
    "Age",
    "Height",
    "Weight",
    "FCVC",
    "NCP",
    "CH2O",
    "FAF",
    "TUE"
]

categorical_features = [
    "Gender",
    "family_history_with_overweight",
    "FAVC",
    "CAEC",
    "SMOKE",
    "SCC",
    "CALC",
    "MTRANS"
]

print("=== FEATURE TYPES ===")
print(f"Numerical features   : {len(numerical_features)}")
print(f"Categorical features : {len(categorical_features)}")
print(
    f"Total features       : {len(numerical_features) + len(categorical_features)}")
print(f"Target               : {TARGET}")

=== FEATURE TYPES ===
Numerical features   : 8
Categorical features : 8
Total features       : 16
Target               : NObeyesdad


In [3]:
# ============================================================
# Cell 3: Data integrity check
# ============================================================

print("=== DATASET SHAPE ===")
print(f"Rows     : {df.shape[0]}")
print(f"Features : {df.shape[1] - 1}")
print(f"Target   : {TARGET}")


# ------------------------------------------------------------
# Target distribution
# ------------------------------------------------------------

print("\n=== TARGET DISTRIBUTION ===")

target_counts = df[TARGET].value_counts()
target_pct = df[TARGET].value_counts(normalize=True) * 100

target_summary = pd.DataFrame({
    "Count": target_counts,
    "Percent": target_pct.round(2)
})

display(target_summary)


# ------------------------------------------------------------
# Missing values
# ------------------------------------------------------------

print("\n=== MISSING VALUES ===")

missing_summary = pd.DataFrame({
    "Missing": df.isna().sum(),
    "Percent": (df.isna().mean() * 100).round(2)
})

display(
    missing_summary[
        missing_summary["Missing"] > 0
    ]
)


# ------------------------------------------------------------
# Duplicate rows
# ------------------------------------------------------------

print("\n=== DUPLICATE ROWS ===")
print(f"Duplicate rows: {df.duplicated().sum()}")


# ------------------------------------------------------------
# Unique values
# ------------------------------------------------------------

print("\n=== UNIQUE VALUES PER FEATURE ===")

unique_summary = pd.DataFrame({
    "Unique values": df.drop(
        columns=TARGET
    ).nunique(dropna=False)
}).sort_values("Unique values")

display(unique_summary)


# ------------------------------------------------------------
# Data types
# ------------------------------------------------------------

print("\n=== DATA TYPES ===")
display(df.dtypes.to_frame("dtype"))

=== DATASET SHAPE ===
Rows     : 2111
Features : 16
Target   : NObeyesdad

=== TARGET DISTRIBUTION ===


,Count,Percent
NObeyesdad,,
Obesity_Type_I,351,16.63
Obesity_Type_III,324,15.35
Obesity_Type_II,297,14.07
Overweight_Level_I,290,13.74
Overweight_Level_II,290,13.74
Normal_Weight,287,13.60
Insufficient_Weight,272,12.88



=== MISSING VALUES ===


,Missing,Percent



=== DUPLICATE ROWS ===
Duplicate rows: 24

=== UNIQUE VALUES PER FEATURE ===


,Unique values
Gender,2
family_history_with_overweight,2
FAVC,2
SMOKE,2
SCC,2
CAEC,4
CALC,4
MTRANS,5
NCP,635
FCVC,810



=== DATA TYPES ===


,dtype
Gender,object
Age,float64
Height,float64
Weight,float64
family_history_with_overweight,object
FAVC,object
FCVC,float64
NCP,float64
CAEC,object
SMOKE,object


In [4]:
# ============================================================
# Cell 4: Duplicate-pattern and feature redundancy checks
# ============================================================

import numpy as np
import pandas as pd
from scipy.stats import chi2_contingency

NUMERIC_CORR_THRESHOLD = 0.85
CRAMERS_V_THRESHOLD = 0.65


# ============================================================
# 1. Check identical feature patterns
# ============================================================

feature_cols = numerical_features + categorical_features

pattern_target_counts = (
    df.groupby(feature_cols, dropna=False)[TARGET]
      .nunique()
)

conflicting_patterns = (pattern_target_counts > 1).sum()

print("=== DUPLICATE / REPEATED PATTERN CHECK ===")
print(f"Exact duplicate rows: {df.duplicated().sum()}")
print(
    "Feature patterns associated with multiple targets:",
    conflicting_patterns
)


# ============================================================
# 2. Numerical ↔ Numerical correlation
# ============================================================

numeric_corr = df[numerical_features].corr(method="pearson").abs()

numeric_pairs = []

for i in range(len(numerical_features)):
    for j in range(i + 1, len(numerical_features)):

        f1 = numerical_features[i]
        f2 = numerical_features[j]

        corr = numeric_corr.loc[f1, f2]

        if corr >= NUMERIC_CORR_THRESHOLD:
            numeric_pairs.append({
                "Feature 1": f1,
                "Feature 2": f2,
                "|Pearson r|": corr
            })

numeric_pairs_df = pd.DataFrame(numeric_pairs)

print(
    f"\n=== HIGHLY CORRELATED NUMERICAL PAIRS "
    f"(|r| >= {NUMERIC_CORR_THRESHOLD}) ==="
)

if len(numeric_pairs_df) > 0:
    display(
        numeric_pairs_df.sort_values(
            "|Pearson r|",
            ascending=False
        ).style.format({
            "|Pearson r|": "{:.4f}"
        })
    )
else:
    print("No highly correlated numerical pairs found.")


# ============================================================
# 3. Categorical ↔ Categorical Cramér's V
# ============================================================

def cramers_v(x, y):

    x = x.fillna("Unknown").astype(str)
    y = y.fillna("Unknown").astype(str)

    table = pd.crosstab(x, y)

    chi2 = chi2_contingency(table)[0]
    n = table.values.sum()

    r, k = table.shape
    denominator = min(r - 1, k - 1)

    if n == 0 or denominator == 0:
        return 0.0

    return np.sqrt((chi2 / n) / denominator)


categorical_pairs = []

for i in range(len(categorical_features)):
    for j in range(i + 1, len(categorical_features)):

        f1 = categorical_features[i]
        f2 = categorical_features[j]

        v = cramers_v(
            df[f1],
            df[f2]
        )

        if v >= CRAMERS_V_THRESHOLD:
            categorical_pairs.append({
                "Feature 1": f1,
                "Feature 2": f2,
                "Cramer's V": v
            })

categorical_pairs_df = pd.DataFrame(categorical_pairs)

print(
    f"\n=== HIGHLY ASSOCIATED CATEGORICAL PAIRS "
    f"(V >= {CRAMERS_V_THRESHOLD}) ==="
)

if len(categorical_pairs_df) > 0:
    display(
        categorical_pairs_df.sort_values(
            "Cramer's V",
            ascending=False
        ).style.format({
            "Cramer's V": "{:.4f}"
        })
    )
else:
    print("No highly associated categorical pairs found.")

=== DUPLICATE / REPEATED PATTERN CHECK ===
Exact duplicate rows: 24
Feature patterns associated with multiple targets: 0

=== HIGHLY CORRELATED NUMERICAL PAIRS (|r| >= 0.85) ===
No highly correlated numerical pairs found.

=== HIGHLY ASSOCIATED CATEGORICAL PAIRS (V >= 0.65) ===
No highly associated categorical pairs found.


In [5]:
# ============================================================
# Cell 5: Remove exact duplicates and save AVISTA dataset
# ============================================================

from pathlib import Path
import pandas as pd

# ------------------------------------------------------------
# Remove exact duplicate observations
# ------------------------------------------------------------

df_obesity = df.drop_duplicates().reset_index(drop=True)

print("=== CLEANED DATASET ===")
print(f"Original rows       : {len(df)}")
print(f"Duplicates removed  : {len(df) - len(df_obesity)}")
print(f"Final rows          : {len(df_obesity)}")
print(f"Features            : {df_obesity.shape[1] - 1}")
print(f"Target              : {TARGET}")


# ------------------------------------------------------------
# Save dataset for AVISTA
# ------------------------------------------------------------

data_dir = Path("data")
data_dir.mkdir(exist_ok=True)

avista_path = data_dir / "uci_obesity_avista.csv"

df_obesity.to_csv(
    avista_path,
    index=False
)

# Verify saved file
df_check = pd.read_csv(avista_path)

print("\n=== SAVED AVISTA FILE ===")
print(f"Saved to      : {avista_path}")
print(f"Rows          : {df_check.shape[0]}")
print(f"Features      : {df_check.shape[1] - 1}")
print(f"Total columns : {df_check.shape[1]}")
print(f"Target        : {TARGET}")
print(f"Duplicates    : {df_check.duplicated().sum()}")

=== CLEANED DATASET ===
Original rows       : 2111
Duplicates removed  : 24
Final rows          : 2087
Features            : 16
Target              : NObeyesdad

=== SAVED AVISTA FILE ===
Saved to      : data\uci_obesity_avista.csv
Rows          : 2087
Features      : 16
Total columns : 17
Target        : NObeyesdad
Duplicates    : 0


In [6]:
# ============================================================
# Cell 6: Stratified 70/10/20 train-validation-test split
# ============================================================

from sklearn.model_selection import train_test_split

RANDOM_SEED = 42

# Separate predictors and target
X = df_obesity.drop(columns=TARGET)
y = df_obesity[TARGET]

# ------------------------------------------------------------
# Step 1: 70% train / 30% temporary
# ------------------------------------------------------------

X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=RANDOM_SEED,
    stratify=y
)

# ------------------------------------------------------------
# Step 2: Split temporary 30% into
# 10% validation / 20% test
# ------------------------------------------------------------

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=2/3,
    random_state=RANDOM_SEED,
    stratify=y_temp
)

# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------

print("=== DATA SPLIT ===")
print(f"Random seed : {RANDOM_SEED}")
print(
    f"Training    : {len(X_train)} ({len(X_train)/len(df_obesity)*100:.2f}%)")
print(f"Validation  : {len(X_val)} ({len(X_val)/len(df_obesity)*100:.2f}%)")
print(f"Test        : {len(X_test)} ({len(X_test)/len(df_obesity)*100:.2f}%)")

print("\n=== CLASS DISTRIBUTION ===")

for name, target in [
    ("Training", y_train),
    ("Validation", y_val),
    ("Test", y_test)
]:
    print(f"\n{name}")

    summary = pd.DataFrame({
        "Count": target.value_counts(),
        "Percent": (
            target.value_counts(normalize=True) * 100
        ).round(2)
    })

    display(summary)

=== DATA SPLIT ===
Random seed : 42
Training    : 1460 (69.96%)
Validation  : 209 (10.01%)
Test        : 418 (20.03%)

=== CLASS DISTRIBUTION ===

Training


,Count,Percent
NObeyesdad,,
Obesity_Type_I,245,16.78
Obesity_Type_III,227,15.55
Obesity_Type_II,208,14.25
Overweight_Level_II,203,13.90
Normal_Weight,197,13.49
Overweight_Level_I,193,13.22
Insufficient_Weight,187,12.81



Validation


,Count,Percent
NObeyesdad,,
Obesity_Type_I,35,16.75
Obesity_Type_III,32,15.31
Obesity_Type_II,30,14.35
Overweight_Level_II,29,13.88
Overweight_Level_I,28,13.40
Normal_Weight,28,13.40
Insufficient_Weight,27,12.92



Test


,Count,Percent
NObeyesdad,,
Obesity_Type_I,71,16.99
Obesity_Type_III,65,15.55
Obesity_Type_II,59,14.11
Overweight_Level_II,58,13.88
Normal_Weight,57,13.64
Overweight_Level_I,55,13.16
Insufficient_Weight,53,12.68


In [7]:
# ============================================================
# Cell 7: AVISTA-equivalent mixed preprocessing
# Fit on TRAIN only
# ============================================================

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# ------------------------------------------------------------
# Numerical preprocessing
# Median imputation + scaling
# ------------------------------------------------------------

numerical_pipeline = Pipeline([
    (
        "imputer",
        SimpleImputer(strategy="median")
    ),
    (
        "scaler",
        StandardScaler()
    )
])

# ------------------------------------------------------------
# Categorical preprocessing
# Missing -> Unknown + one-hot encoding
# ------------------------------------------------------------

categorical_pipeline = Pipeline([
    (
        "imputer",
        SimpleImputer(
            strategy="constant",
            fill_value="Unknown"
        )
    ),
    (
        "onehot",
        OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=False
        )
    )
])

# ------------------------------------------------------------
# Combined preprocessing
# ------------------------------------------------------------

preprocessor = ColumnTransformer([
    (
        "numerical",
        numerical_pipeline,
        numerical_features
    ),
    (
        "categorical",
        categorical_pipeline,
        categorical_features
    )
])

# ------------------------------------------------------------
# Fit ONLY on 70% training partition
# ------------------------------------------------------------

X_train_processed = preprocessor.fit_transform(X_train)

# Transform validation/test using train-fitted preprocessing
X_val_processed = preprocessor.transform(X_val)
X_test_processed = preprocessor.transform(X_test)

print("=== PROCESSED DATA ===")
print(f"Train      : {X_train_processed.shape}")
print(f"Validation : {X_val_processed.shape}")
print(f"Test       : {X_test_processed.shape}")

print("\nPreprocessing fitted on training data only.")

=== PROCESSED DATA ===
Train      : (1460, 31)
Validation : (209, 31)
Test       : (418, 31)

Preprocessing fitted on training data only.


In [8]:
# ============================================================
# Cell 8: Multiclass SMOTE Moderate (0.5)
# Apply to TRAIN only
# ============================================================

import numpy as np
from imblearn.over_sampling import SMOTE

SMOTE_RATIO = 0.5
SMOTE_SEED = 42

# ------------------------------------------------------------
# Class counts before SMOTE
# ------------------------------------------------------------

class_counts = y_train.value_counts()

majority_count = class_counts.max()

target_count = int(
    np.floor(SMOTE_RATIO * majority_count)
)

print("=== BEFORE SMOTE ===")
print(class_counts)

print(f"\nMajority class count : {majority_count}")
print(f"Moderate 0.5 target  : {target_count}")

# ------------------------------------------------------------
# Oversample only classes below the 0.5 threshold
# ------------------------------------------------------------

sampling_strategy = {
    cls: target_count
    for cls, count in class_counts.items()
    if count < target_count
}

print("\nSampling strategy:")
print(sampling_strategy)

# ------------------------------------------------------------
# Apply to TRAIN only
# ------------------------------------------------------------

if sampling_strategy:

    smote = SMOTE(
        sampling_strategy=sampling_strategy,
        random_state=SMOTE_SEED
    )

    X_train_final, y_train_final = smote.fit_resample(
        X_train_processed,
        y_train
    )

    print("\nSMOTE applied.")

else:

    X_train_final = X_train_processed
    y_train_final = y_train.copy()

    print("\nSMOTE not required.")

# ------------------------------------------------------------
# Final training distribution
# ------------------------------------------------------------

print("\n=== AFTER SMOTE ===")
print(y_train_final.value_counts())

print(f"\nOriginal training N  : {len(y_train)}")
print(f"Effective training N : {len(y_train_final)}")

=== BEFORE SMOTE ===
NObeyesdad
Obesity_Type_I         245
Obesity_Type_III       227
Obesity_Type_II        208
Overweight_Level_II    203
Normal_Weight          197
Overweight_Level_I     193
Insufficient_Weight    187
Name: count, dtype: int64

Majority class count : 245
Moderate 0.5 target  : 122

Sampling strategy:
{}

SMOTE not required.

=== AFTER SMOTE ===
NObeyesdad
Obesity_Type_I         245
Obesity_Type_III       227
Obesity_Type_II        208
Overweight_Level_II    203
Normal_Weight          197
Overweight_Level_I     193
Insufficient_Weight    187
Name: count, dtype: int64

Original training N  : 1460
Effective training N : 1460


In [9]:
# ============================================================
# Cell 9: 5-Fold CV
# Logistic Regression + Random Forest + XGBoost
# ============================================================

import time
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import (
    OneHotEncoder,
    StandardScaler,
    LabelEncoder
)

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score
)

from imblearn.over_sampling import SMOTE


RANDOM_SEED = 42
N_FOLDS = 5
SMOTE_RATIO = 0.5

skf = StratifiedKFold(
    n_splits=N_FOLDS,
    shuffle=True,
    random_state=RANDOM_SEED
)

label_encoder = LabelEncoder()
label_encoder.fit(y_train)

results = {
    "Logistic Regression": [],
    "Random Forest": [],
    "XGBoost": []
}


for fold, (train_idx, val_idx) in enumerate(
    skf.split(X_train, y_train),
    start=1
):

    print(f"Running fold {fold}...")

    X_fold_train = X_train.iloc[train_idx].copy()
    y_fold_train = y_train.iloc[train_idx].copy()

    X_fold_val = X_train.iloc[val_idx].copy()
    y_fold_val = y_train.iloc[val_idx].copy()

    # ========================================================
    # Preprocessing fitted on FOLD TRAIN only
    # ========================================================

    numerical_pipeline = Pipeline([
        (
            "imputer",
            SimpleImputer(strategy="median")
        ),
        (
            "scaler",
            StandardScaler()
        )
    ])

    categorical_pipeline = Pipeline([
        (
            "imputer",
            SimpleImputer(
                strategy="constant",
                fill_value="Unknown"
            )
        ),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            )
        )
    ])

    fold_preprocessor = ColumnTransformer([
        (
            "numerical",
            numerical_pipeline,
            numerical_features
        ),
        (
            "categorical",
            categorical_pipeline,
            categorical_features
        )
    ])

    X_fold_train_processed = (
        fold_preprocessor.fit_transform(X_fold_train)
    )

    X_fold_val_processed = (
        fold_preprocessor.transform(X_fold_val)
    )

    # ========================================================
    # SMOTE Moderate 0.5 on FOLD TRAIN only
    # ========================================================

    counts = y_fold_train.value_counts()
    majority_count = counts.max()

    target_count = int(
        np.floor(SMOTE_RATIO * majority_count)
    )

    sampling_strategy = {
        cls: target_count
        for cls, count in counts.items()
        if count < target_count
    }

    if sampling_strategy:

        smote = SMOTE(
            sampling_strategy=sampling_strategy,
            random_state=RANDOM_SEED
        )

        X_fit, y_fit = smote.fit_resample(
            X_fold_train_processed,
            y_fold_train
        )

    else:

        X_fit = X_fold_train_processed
        y_fit = y_fold_train.copy()

    # ========================================================
    # Logistic Regression
    # ========================================================

    lr = LogisticRegression(
        random_state=RANDOM_SEED,
        max_iter=1000
    )

    start = time.perf_counter()
    lr.fit(X_fit, y_fit)
    runtime = time.perf_counter() - start

    pred = lr.predict(X_fold_val_processed)
    prob = lr.predict_proba(X_fold_val_processed)

    results["Logistic Regression"].append({
        "Fold": fold,
        "Accuracy": accuracy_score(y_fold_val, pred),
        "Macro-F1": f1_score(
            y_fold_val, pred, average="macro"
        ),
        "Precision": precision_score(
            y_fold_val, pred,
            average="macro",
            zero_division=0
        ),
        "Recall": recall_score(
            y_fold_val, pred,
            average="macro",
            zero_division=0
        ),
        "ROC-AUC": roc_auc_score(
            y_fold_val,
            prob,
            multi_class="ovr",
            average="macro"
        ),
        "Train N": len(y_fold_train),
        "Effective N": len(y_fit),
        "Runtime (sec)": runtime
    })

    # ========================================================
    # Random Forest
    # ========================================================

    rf = RandomForestClassifier(
        n_estimators=100,
        random_state=RANDOM_SEED,
        n_jobs=-1
    )

    start = time.perf_counter()
    rf.fit(X_fit, y_fit)
    runtime = time.perf_counter() - start

    pred = rf.predict(X_fold_val_processed)
    prob = rf.predict_proba(X_fold_val_processed)

    results["Random Forest"].append({
        "Fold": fold,
        "Accuracy": accuracy_score(y_fold_val, pred),
        "Macro-F1": f1_score(
            y_fold_val, pred, average="macro"
        ),
        "Precision": precision_score(
            y_fold_val, pred,
            average="macro",
            zero_division=0
        ),
        "Recall": recall_score(
            y_fold_val, pred,
            average="macro",
            zero_division=0
        ),
        "ROC-AUC": roc_auc_score(
            y_fold_val,
            prob,
            multi_class="ovr",
            average="macro"
        ),
        "Train N": len(y_fold_train),
        "Effective N": len(y_fit),
        "Runtime (sec)": runtime
    })

    # ========================================================
    # XGBoost
    # ========================================================

    y_fit_xgb = label_encoder.transform(y_fit)
    y_val_xgb = label_encoder.transform(y_fold_val)

    xgb = XGBClassifier(
        n_estimators=100,
        max_depth=6,
        learning_rate=0.3,
        random_state=RANDOM_SEED,
        eval_metric="mlogloss",
        n_jobs=-1
    )

    start = time.perf_counter()
    xgb.fit(X_fit, y_fit_xgb)
    runtime = time.perf_counter() - start

    pred = xgb.predict(X_fold_val_processed)
    prob = xgb.predict_proba(X_fold_val_processed)

    results["XGBoost"].append({
        "Fold": fold,
        "Accuracy": accuracy_score(y_val_xgb, pred),
        "Macro-F1": f1_score(
            y_val_xgb, pred, average="macro"
        ),
        "Precision": precision_score(
            y_val_xgb, pred,
            average="macro",
            zero_division=0
        ),
        "Recall": recall_score(
            y_val_xgb, pred,
            average="macro",
            zero_division=0
        ),
        "ROC-AUC": roc_auc_score(
            y_val_xgb,
            prob,
            multi_class="ovr",
            average="macro"
        ),
        "Train N": len(y_fold_train),
        "Effective N": len(y_fit),
        "Runtime (sec)": runtime
    })


# ============================================================
# Summary
# ============================================================

metrics = [
    "Accuracy",
    "Macro-F1",
    "Precision",
    "Recall",
    "ROC-AUC"
]

for model_name, model_results in results.items():

    result_df = pd.DataFrame(model_results)

    print(
        f"\n=== {model_name.upper()} — 5-FOLD CV SUMMARY ==="
    )

    for metric in metrics:
        print(
            f"{metric:10s}: "
            f"{result_df[metric].mean():.6f} "
            f"± {result_df[metric].std():.6f}"
        )

Running fold 1...
Running fold 2...
Running fold 3...
Running fold 4...
Running fold 5...

=== LOGISTIC REGRESSION — 5-FOLD CV SUMMARY ===
Accuracy  : 0.865753 ± 0.025096
Macro-F1  : 0.858218 ± 0.024289
Precision : 0.863357 ± 0.024842
Recall    : 0.862115 ± 0.023940
ROC-AUC   : 0.982222 ± 0.003855

=== RANDOM FOREST — 5-FOLD CV SUMMARY ===
Accuracy  : 0.934247 ± 0.002865
Macro-F1  : 0.931797 ± 0.002568
Precision : 0.934805 ± 0.003447
Recall    : 0.931729 ± 0.002655
ROC-AUC   : 0.992300 ± 0.001182

=== XGBOOST — 5-FOLD CV SUMMARY ===
Accuracy  : 0.954795 ± 0.016495
Macro-F1  : 0.952908 ± 0.017089
Precision : 0.954869 ± 0.016158
Recall    : 0.952986 ± 0.016934
ROC-AUC   : 0.997737 ± 0.000878


In [10]:
# ============================================================
# Cell 10: Final evaluation on untouched 20% test set
# Logistic Regression + Random Forest + XGBoost
# ============================================================

import time
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import (
    OneHotEncoder,
    StandardScaler,
    LabelEncoder
)

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score
)

from imblearn.over_sampling import SMOTE


RANDOM_SEED = 42
SMOTE_RATIO = 0.5


# ------------------------------------------------------------
# 1. Fit preprocessing on full 70% training set only
# ------------------------------------------------------------

final_preprocessor = ColumnTransformer([
    (
        "numerical",
        Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler())
        ]),
        numerical_features
    ),
    (
        "categorical",
        Pipeline([
            (
                "imputer",
                SimpleImputer(
                    strategy="constant",
                    fill_value="Unknown"
                )
            ),
            (
                "onehot",
                OneHotEncoder(
                    handle_unknown="ignore",
                    sparse_output=False
                )
            )
        ]),
        categorical_features
    )
])

X_train_processed = final_preprocessor.fit_transform(X_train)
X_test_processed = final_preprocessor.transform(X_test)


# ------------------------------------------------------------
# 2. SMOTE Moderate (0.5) on training only
# ------------------------------------------------------------

counts = y_train.value_counts()
majority_count = counts.max()

target_count = int(
    np.floor(SMOTE_RATIO * majority_count)
)

sampling_strategy = {
    cls: target_count
    for cls, count in counts.items()
    if count < target_count
}

if sampling_strategy:

    smote = SMOTE(
        sampling_strategy=sampling_strategy,
        random_state=RANDOM_SEED
    )

    X_train_final, y_train_final = smote.fit_resample(
        X_train_processed,
        y_train
    )

else:

    X_train_final = X_train_processed
    y_train_final = y_train.copy()


print("=== FINAL TRAINING DATA ===")
print(f"Original training N  : {len(y_train)}")
print(f"Effective training N : {len(y_train_final)}")
print(f"Test N               : {len(y_test)}")


# ------------------------------------------------------------
# 3. Label encoding for XGBoost
# ------------------------------------------------------------

label_encoder = LabelEncoder()
label_encoder.fit(y_train)

y_train_xgb = label_encoder.transform(y_train_final)
y_test_xgb = label_encoder.transform(y_test)


# ------------------------------------------------------------
# 4. Models
# ------------------------------------------------------------

models = {
    "Logistic Regression": LogisticRegression(
        random_state=RANDOM_SEED,
        max_iter=1000
    ),

    "Random Forest": RandomForestClassifier(
        n_estimators=100,
        random_state=RANDOM_SEED,
        n_jobs=-1
    ),

    "XGBoost": XGBClassifier(
        n_estimators=100,
        max_depth=6,
        learning_rate=0.3,
        random_state=RANDOM_SEED,
        eval_metric="mlogloss",
        n_jobs=-1
    )
}


# ------------------------------------------------------------
# 5. Final test evaluation
# ------------------------------------------------------------

results = []

for model_name, model in models.items():

    start = time.perf_counter()

    if model_name == "XGBoost":

        model.fit(
            X_train_final,
            y_train_xgb
        )

        runtime = time.perf_counter() - start

        y_pred = model.predict(X_test_processed)
        y_prob = model.predict_proba(X_test_processed)

        y_true = y_test_xgb

    else:

        model.fit(
            X_train_final,
            y_train_final
        )

        runtime = time.perf_counter() - start

        y_pred = model.predict(X_test_processed)
        y_prob = model.predict_proba(X_test_processed)

        y_true = y_test

    results.append({
        "Model": model_name,

        "Accuracy": accuracy_score(
            y_true,
            y_pred
        ),

        "Macro-F1": f1_score(
            y_true,
            y_pred,
            average="macro"
        ),

        "Precision": precision_score(
            y_true,
            y_pred,
            average="macro",
            zero_division=0
        ),

        "Recall": recall_score(
            y_true,
            y_pred,
            average="macro",
            zero_division=0
        ),

        "ROC-AUC": roc_auc_score(
            y_true,
            y_prob,
            multi_class="ovr",
            average="macro"
        ),

        "Training N": len(y_train),
        "Effective N": len(y_train_final),
        "Test N": len(y_test),
        "Runtime (sec)": runtime
    })


# ------------------------------------------------------------
# 6. Results
# ------------------------------------------------------------

test_results = pd.DataFrame(results)

display(
    test_results.style.format({
        "Accuracy": "{:.6f}",
        "Macro-F1": "{:.6f}",
        "Precision": "{:.6f}",
        "Recall": "{:.6f}",
        "ROC-AUC": "{:.6f}",
        "Runtime (sec)": "{:.4f}"
    })
)

=== FINAL TRAINING DATA ===
Original training N  : 1460
Effective training N : 1460
Test N               : 418


,Model,Accuracy,Macro-F1,Precision,Recall,ROC-AUC,Training N,Effective N,Test N,Runtime (sec)
0,Logistic Regression,0.894737,0.891982,0.893111,0.892911,0.986106,1460,1460,418,0.0753
1,Random Forest,0.942584,0.940953,0.943806,0.939610,0.994385,1460,1460,418,0.2912
2,XGBoost,0.968900,0.968381,0.968784,0.968384,0.999251,1460,1460,418,0.6918


In [12]:
# ============================================================
# Cell 11: AutoGluon — UCI Obesity Dataset
# Logistic Regression + Random Forest + XGBoost
# Same raw 70/10/20 split + 5-fold bagging
# ============================================================

import time
import pandas as pd

from autogluon.tabular import TabularPredictor

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score
)

TARGET = "NObeyesdad"


# ============================================================
# 1. Build RAW datasets using the SAME split rows
#
# IMPORTANT:
# - No manual one-hot encoding
# - No StandardScaler
# - No manual imputation
# - No SMOTE
# - AutoGluon handles preprocessing internally
# ============================================================

ag_train = X_train.copy()
ag_train[TARGET] = y_train.values

ag_val = X_val.copy()
ag_val[TARGET] = y_val.values

ag_test = X_test.copy()
ag_test[TARGET] = y_test.values


print("=== AUTOGLUON DATA ===")
print(f"Training    : {len(ag_train)}")
print(f"Validation  : {len(ag_val)}")
print(f"Test        : {len(ag_test)}")

print("\nTraining class distribution:")
print(
    ag_train[TARGET]
    .value_counts()
)


# ============================================================
# 2. Train AutoGluon
#
# LR  = Logistic Regression / LinearModel
# RF  = Random Forest
# XGB = XGBoost
# ============================================================

start_time = time.perf_counter()

predictor_obesity = TabularPredictor(
    label=TARGET,
    problem_type="multiclass",
    eval_metric="accuracy",
    path="AutogluonModels/obesity_LR_RF_XGB_5fold"
).fit(

    # Same 70% training observations
    train_data=ag_train,

    # Same 10% validation observations
    tuning_data=ag_val,

    # Restrict to same three model families
    hyperparameters={
        "LR": {},
        "RF": {},
        "XGB": {}
    },

    # 5-fold bagging
    num_bag_folds=5,
    num_bag_sets=1,

    # No stacking
    num_stack_levels=0,

    # Use supplied 10% validation set as holdout
    use_bag_holdout=True,

    # No weighted ensemble
    fit_weighted_ensemble=False,
    fit_full_last_level_weighted_ensemble=False
)

total_training_runtime = (
    time.perf_counter() - start_time
)


# ============================================================
# 3. AutoGluon leaderboard
# ============================================================

print("\n=== AUTOGLUON LEADERBOARD ===")

leaderboard = predictor_obesity.leaderboard(
    ag_test,
    extra_info=False
)

display(leaderboard)


# ============================================================
# 4. Untouched 20% test set
# ============================================================

X_ag_test = ag_test.drop(columns=TARGET)
y_ag_test = ag_test[TARGET]

class_labels = list(
    predictor_obesity.class_labels
)

print("\nAutoGluon class order:")
print(class_labels)


# ============================================================
# 5. Evaluate each model separately
# ============================================================

results = []

for model_name in leaderboard["model"].tolist():

    start_pred = time.perf_counter()

    y_pred = predictor_obesity.predict(
        X_ag_test,
        model=model_name
    )

    y_prob = predictor_obesity.predict_proba(
        X_ag_test,
        model=model_name
    )

    prediction_runtime = (
        time.perf_counter() - start_pred
    )

    # Ensure probabilities follow AutoGluon's class order
    if isinstance(y_prob, pd.DataFrame):
        y_prob_ordered = y_prob[
            class_labels
        ].values
    else:
        y_prob_ordered = y_prob

    results.append({

        "Model": model_name,

        "Accuracy": accuracy_score(
            y_ag_test,
            y_pred
        ),

        "Macro-F1": f1_score(
            y_ag_test,
            y_pred,
            average="macro"
        ),

        "Precision": precision_score(
            y_ag_test,
            y_pred,
            average="macro",
            zero_division=0
        ),

        "Recall": recall_score(
            y_ag_test,
            y_pred,
            average="macro",
            zero_division=0
        ),

        "ROC-AUC": roc_auc_score(
            y_ag_test,
            y_prob_ordered,
            labels=class_labels,
            multi_class="ovr",
            average="macro"
        ),

        "Training N": len(ag_train),

        "Validation N": len(ag_val),

        "Test N": len(ag_test),

        "CV Folds": 5,

        "SMOTE": "No",

        "Prediction Runtime (sec)": prediction_runtime
    })


# ============================================================
# 6. Final results table
# ============================================================

ag_obesity_results = pd.DataFrame(results)

print(
    "\n=== AUTOGLUON — OBESITY FINAL TEST RESULTS ==="
)

display(
    ag_obesity_results.style.format({
        "Accuracy": "{:.6f}",
        "Macro-F1": "{:.6f}",
        "Precision": "{:.6f}",
        "Recall": "{:.6f}",
        "ROC-AUC": "{:.6f}",
        "Prediction Runtime (sec)": "{:.4f}"
    })
)


# ============================================================
# 7. Overall AutoGluon runtime
# ============================================================

print(
    f"\nTotal AutoGluon training runtime: "
    f"{total_training_runtime:.3f} sec"
)

Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.6.3
Python Version:     3.12.9
Operating System:   Windows
Platform Machine:   AMD64
Platform Version:   10.0.26200
CPU Count:          12
Pytorch Version:    2.6.0+cu118
CUDA Version:       11.8
GPU Memory:         GPU 0: 8.00/8.00 GB
Total GPU Memory:   Free: 8.00 GB, Allocated: 0.00 GB, Total: 8.00 GB
GPU Count:          1
Memory Avail:       19.81 GB / 47.93 GB (41.3%)
Disk Space Avail:   267.28 GB / 610.93 GB (43.7%)
No presets specified! To achieve strong results with AutoGluon, it is recommended to use the available presets. Defaulting to `'medium'`...
	Recommended Presets (For more details refer to https://auto.gluon.ai/stable/tutorials/tabular/tabular-essentials.html#presets):
	presets='extreme'  : Use this if you have a GPU. The go-to preset for best results, and the one to use for benchmark comparisons. New in v1.6: far better than 'best' on datasets <100000 samples by u

=== AUTOGLUON DATA ===
Training    : 1460
Validation  : 209
Test        : 418

Training class distribution:
NObeyesdad
Obesity_Type_I         245
Obesity_Type_III       227
Obesity_Type_II        208
Overweight_Level_II    203
Normal_Weight          197
Overweight_Level_I     193
Insufficient_Weight    187
Name: count, dtype: int64


Beginning AutoGluon training ...
AutoGluon will save models to "AutogluonModels\obesity_LR_RF_XGB_5fold"
Train Data Rows:    1460
Train Data Columns: 16
Tuning Data Rows:    209
Tuning Data Columns: 16
Label Column:       NObeyesdad
Problem Type:       multiclass
Preprocessing data...
Train Data Class Count: 7
Using Feature Generators to preprocess the data ...
Fitting AutoMLPipelineFeatureGenerator...
	Available Memory:                    20255.61 MB
	Train Data (Original)  Memory Usage: 0.80 MB (0.0% of available memory)
	Inferring data type of each feature based on column values. Set feature_metadata_in to manually specify special dtypes of the features.
	Stage 1 Generators:
		Fitting AsTypeFeatureGenerator...
			Note: Converting 5 features to boolean dtype as they only contain 2 unique values.
	Stage 2 Generators:
		Fitting FillNaFeatureGenerator...
	Stage 3 Generators:
		Fitting IdentityFeatureGenerator...
		Fitting CategoryFeatureGenerator...
			Fitting CategoryMemoryMinimizeFeat


=== AUTOGLUON LEADERBOARD ===


,model,score_test,score_val,eval_metric,pred_time_test,pred_time_val,fit_time,pred_time_test_marginal,pred_time_val_marginal,fit_time_marginal,stack_level,can_infer,fit_order
0,XGBoost_BAG_L1,0.971292,0.947368,accuracy,0.476272,0.671053,3.976959,0.476272,0.671053,3.976959,1,True,2
1,RandomForest_BAG_L1,0.954545,0.952153,accuracy,0.160496,0.158406,0.871736,0.160496,0.158406,0.871736,1,True,1
2,LinearModel_BAG_L1,0.904306,0.875598,accuracy,0.105281,0.147027,4.086481,0.105281,0.147027,4.086481,1,True,3



AutoGluon class order:
['Insufficient_Weight', 'Normal_Weight', 'Obesity_Type_I', 'Obesity_Type_II', 'Obesity_Type_III', 'Overweight_Level_I', 'Overweight_Level_II']

=== AUTOGLUON — OBESITY FINAL TEST RESULTS ===


,Model,Accuracy,Macro-F1,Precision,Recall,ROC-AUC,Training N,Validation N,Test N,CV Folds,SMOTE,Prediction Runtime (sec)
0,XGBoost_BAG_L1,0.971292,0.970215,0.970178,0.970536,0.998883,1460,209,418,5,No,0.9307
1,RandomForest_BAG_L1,0.954545,0.953147,0.954848,0.952372,0.997132,1460,209,418,5,No,0.3294
2,LinearModel_BAG_L1,0.904306,0.901550,0.903122,0.902771,0.984476,1460,209,418,5,No,0.1162



Total AutoGluon training runtime: 18.432 sec
